In [1]:
import PIL

import os
import base64
from io import BytesIO

import torch

import datasets
from transformers import Qwen3_5ForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

/Data/joao.giordani-donasolo/multigen/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
ds_qa = datasets.load_dataset("flaviagiammarino/path-vqa", cache_dir="huggingface/")
ds_pm = datasets.load_dataset("FreedomIntelligence/PubMedVision", "PubMedVision_Alignment_VQA", cache_dir="huggingface/")

In [3]:
def get_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

def get_lora_config():
    """Build LoRA config from the config dict."""
    return LoraConfig(
        r=64,
        lora_alpha=64,
        lora_dropout=0.2,
        # target_modules=[
        #     "q_proj", "w_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
        #     "proj", "qkv", "linear_fc1", "linear_fc2"
        # ],
        target_modules="all-linear", 
        task_type=TaskType.CAUSAL_LM,
        bias="none",
    )

def load_model(
    model_name: str, 
    cache_dir: str = "huggingface",
    quantize: bool = True,
    peft: bool = False
):  
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        model_name, 
        cache_dir=cache_dir,
        quantization_config=get_bnb_config() if quantize else None
    ).to(DEVICE)
    processor = AutoProcessor.from_pretrained(model_name, cache_dir=cache_dir)

    if quantize:
        model = prepare_model_for_kbit_training(model)

    if peft:
        model = get_peft_model(model, peft_config=get_lora_config())

    return model, processor


model, processor = load_model("Qwen/Qwen3.5-0.8B", peft=True)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 1320.11it/s]


In [4]:
model.print_trainable_parameters()

trainable params: 53,383,168 || all params: 906,369,088 || trainable%: 5.8898


In [5]:
processor.tokenizer.padding_side

'right'

In [6]:
def preprocess_dataset(example):
    
    # buf = BytesIO()
    img = example["image"]
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    # img.save(buf, format="JPEG", quality=85)  # JPEG >> PNG speed
    # img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    
    return {
        "prompt": [{"role": "user", "content": [
            {"type": "text", "text": example["question"]},
            {"type": "image"},
        ]}],
        "completion": [{"role": "assistant", "content": example["answer"]}],
        "images": [example["image"]]
    }

ds_qa_train = ds_qa["train"].map(
    preprocess_dataset,
    remove_columns=ds_qa["train"].column_names,
    num_proc=os.cpu_count(),
    writer_batch_size=500,
)

In [7]:
def collate_fn(examples):
    texts, images = [], []

    for example in examples:
        messages = example["prompt"] + example["completion"]  # full conversation
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
        images.append(example["images"][0])

    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )
    batch["labels"] = batch["input_ids"].clone()
    return batch

In [13]:
config = SFTConfig(
    output_dir="results/baseline",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=32,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    # eval_steps=200,
    # eval_strategy="steps",
    # save_total_limit=2,
    # load_best_model_at_end=True,
    # metric_for_best_model="eval_loss",
    report_to="wandb",
    run_name="multigen-baseline",
    max_length=512,
    dataset_text_field="text",
    seed=42,
)


trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    train_dataset=ds_qa_train,
    data_collator=collate_fn,
    args=config
)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /users/eleves-a/2025/joao.giordani-donasolo/.netrc.
wandb: Currently logged in as: jpdonasolo (mtomita143-ecole-polytehcnique) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f7971729400>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f7b1c597860, execution_count=13 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f7b1c594d40, raw_cell="config = SFTConfig(
    output_dir="results/baseli.." transformed_cell="config = SFTConfig(
    output_dir="results/baseli.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22766d2d706f6c79746563682d61696e227d/Data/joao.giordani-donasolo/multigen/main.ipynb#X21sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:



img = ds_qa["train"]["image"][0]
buf = BytesIO()
if img.mode not in ("RGB", "RGBA", "L"):
    img = img.convert("RGB")
img.save(buf, format="PNG")
img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")



messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful assistant."}
        ]
    },
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": img_b64,
            },
            # {"type": "text", "text": ds["train"][0]["question"]},
            {"type": "text", "text": "describe this image"},
        ],
    }
]
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}


generated_ids = model.generate(**inputs, max_new_tokens=64)
output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print(output_text)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


system
You are a helpful assistant.
user
describe this image
assistant
<think>

</think>

This image is a photomicrograph of a tissue section stained with hematoxylin and eosin (H&E). The image displays a section of connective tissue, likely from a lymph node or a similar lymphoid organ, characterized by a dense collection of lymphocytes.

The image highlights two distinct structures:


In [17]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_grid_thw'])

In [18]:
inputs["pixel_values"].shape

torch.Size([1600, 1536])